# 08 Pipeline, tests that fail, and economics
Turn the notebooks into a pipeline, prove each test can fail, and put a price on the work. `pipeline.run_baseline` runs notebooks 03 to 06 in one call.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
import time
tol = P["reconcile"]["tolerance"]
t0 = time.perf_counter(); B = pipeline.run_baseline(RDIR, tol); secs = time.perf_counter() - t0
rec = B["rec"]
print(f"Pipeline: {len(rec)} filings in {secs:.2f} s ({secs / max(len(rec), 1):.3f} s per filing); admitted {int(rec['admitted'].sum())}")

In [ ]:
# A test is only evidence if it can fail. Remove the totals rule and watch admissions change.
broken = pipeline.run_baseline(RDIR, tol, rules=[clean.rule_cash])["rec"]
print("Admitted with the totals rule:", int(rec["admitted"].sum()), "| without it:", int(broken["admitted"].sum()))
if P["data"]["use_fixture"]:
    assert int(broken["admitted"].sum()) < int(rec["admitted"].sum()), "on the fixture, dropping the totals rule must lose an admission"

In [ ]:
# Five-line unit economics. ASSUMPTIONS: replace with your own, justify them, and show x0.5 and x2.
compute_usd_per_hour = 0.10        # e.g. a laptop or a small cloud VM
analyst_hours_per_bdc, analyst_rate = 2.0, 120.0
n = len(rec)
per_filing = secs / max(n, 1) / 3600 * compute_usd_per_hour
print(f"Cost per completed task: ${per_filing:.2e} of compute per filing (${1000 * per_filing:.4f} per 1,000 filings)")
print(f"Latency: {secs:.2f} s for the release; {secs / max(n, 1):.3f} s per filing on average")
print(f"Quality: {rec['admitted'].mean():.0%} of filings admitted")
print(pd.DataFrame([{"scenario": f"x{k}", "analyst_hours": analyst_hours_per_bdc * k * n,
                     "analyst_cost_usd": analyst_hours_per_bdc * k * n * analyst_rate} for k in (0.5, 1.0, 2.0)]).to_string(index=False))

In [ ]:
dvc_example = """stages:
  acquire:   {cmd: python scripts/acquire.py,   deps: [scripts/acquire.py, params.yaml], outs: [data/raw]}
  parse:     {cmd: python scripts/parse.py,     deps: [scripts/parse.py, data/raw], outs: [outputs/canonical]}
  clean:     {cmd: python scripts/clean.py,     deps: [scripts/clean.py, outputs/canonical], outs: [outputs/cleaned]}
  reconcile: {cmd: python scripts/reconcile.py, deps: [scripts/reconcile.py, outputs/cleaned], params: [reconcile.tolerance],
              outs: [outputs/panel], metrics: [{reports/coverage.json: {cache: false}}]}
"""
(ROOT / "docs" / "dvc.yaml.example").write_text(dvc_example)
print("Wrote docs/dvc.yaml.example; reports/coverage.json is written by report.write_outputs (notebook 06)")

**Your turn.** Move your notebook logic into `scripts/` (one script per stage), copy the example to `dvc.yaml`, and confirm that `dvc repro` rebuilds your panel and reports from the raw release (Case Study 2B, Part 5).